<a href="https://colab.research.google.com/github/kuds/reinforce-tactics/blob/main/notebooks/balance_analysis.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Balance Analysis - Reinforce Tactics

This notebook runs a tournament across **multiple maps spanning small to large sizes**, then measures two balance signals:

1. **First-turn advantage** — P1 vs P2 win rate, broken down by map and bot pairing, with Wilson 95% confidence intervals. Draws are reported separately and as a 0.5 score.
2. **Unit competitiveness** — for each unit, how often it's built, share of gold spent on it, and whether building it correlates with winning.

The default map pool covers 6x6 (beginner, starter), 8x8 (skirmish), 10x10-13x11 (difficult_terrain, funnel_point, corner_points), and 15x15+ (crossroads, the_narrows) so you can see whether first-mover effects and warrior dominance scale with map size. Edit `MAPS` in the config cell to change the pool.

The **Environment Overrides** cell exposes per-player starting gold, per-player free starting unit, unit costs, unit stats, and structure income as configurable knobs so you can re-run the same measurement against candidate balance changes (warrior cost bump, P2 komi, etc.) without editing source.

By default outputs are saved to **Google Drive** so they survive a Colab runtime disconnect — set `USE_GOOGLE_DRIVE = False` in the Storage cell to keep results local.

**Run order:**
1. Setup
2. Configure tournament parameters (maps, games-per-side)
3. Mount Google Drive (or fall back to local storage)
4. Set environment overrides (or leave blank for baseline)
5. Run tournament (writes results + replays to Drive)
6. Analyse first-turn advantage from `GameResult` data
7. Analyse bot-level performance and unit competitiveness from replay JSONs
8. Export combined CSVs + JSON bundle

## Setup

In [ ]:
import os
import subprocess
import sys

IN_COLAB = "google.colab" in sys.modules
REPO_URL = "https://github.com/kuds/reinforce-tactics.git"
REPO_DIR = "reinforce-tactics"

if IN_COLAB:
    print("Running in Google Colab - setting up environment...")
    if not os.path.exists(REPO_DIR):
        subprocess.run(["git", "clone", REPO_URL], check=True)
    else:
        print("Repository already cloned")
    os.chdir(REPO_DIR)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"], check=True)
    print("Dependencies installed")
else:
    if os.path.basename(os.getcwd()) == "notebooks":
        os.chdir("..")

if os.getcwd() not in sys.path:
    sys.path.insert(0, os.getcwd())

print("Working directory:", os.getcwd())

## Configuration

Edit `MAPS`, `GAMES_PER_SIDE`, and bot selection here. Default pool spans **small (6x6) through large (16x16)** maps so you can compare how balance issues vary with map size. With 3 built-in bots and `GAMES_PER_SIDE=4`, the default config plays `3 matchups x 8 maps x 8 games = 192` games.

In [ ]:
from datetime import datetime
from pathlib import Path

# Maps spanning small -> large to measure how balance signals scale with size.
# Tag each map with its (width, height) so analysis tables can group by size.
MAP_POOL = [
    ("maps/1v1/beginner.csv", (6, 6), "small"),
    ("maps/1v1/starter.csv", (6, 6), "small"),
    ("maps/1v1/skirmish.csv", (8, 8), "small"),
    ("maps/1v1/difficult_terrain.csv", (10, 10), "medium"),
    ("maps/1v1/funnel_point.csv", (13, 11), "medium"),
    ("maps/1v1/corner_points.csv", (12, 10), "medium"),
    ("maps/1v1/crossroads.csv", (15, 15), "large"),
    ("maps/1v1/the_narrows.csv", (16, 16), "large"),
]
MAPS = [m[0] for m in MAP_POOL]
MAP_SIZE = {Path(m[0]).stem: m[1] for m in MAP_POOL}
MAP_BUCKET = {Path(m[0]).stem: m[2] for m in MAP_POOL}

# Tournament parameters
GAMES_PER_SIDE = 4
# Raised from 200 because crossroads (15x15) and skirmish (8x8) hit the
# turn limit in 100% of games at 200, drowning out the first-turn-advantage
# signal. 250 resolved skirmish; 300 gives crossroads room to finish too.
MAX_TURNS = 300
CONCURRENT = 1

RUN_TS = datetime.now().strftime("%Y%m%d_%H%M%S")

USE_BUILTIN_ONLY = True

missing = [m for m in MAPS if not Path(m).exists()]
if missing:
    raise FileNotFoundError(f"Maps not found: {missing}")
print(f"Configured {len(MAPS)} maps across sizes: {sorted(set(MAP_BUCKET.values()))}")

## Storage

When running in Colab, results are saved to Google Drive so they survive a runtime disconnect. Set `USE_GOOGLE_DRIVE = False` to keep things on the local Colab/local filesystem instead.

Final output path: `{SAVE_DIR}/balance_analysis_{RUN_TAG}_{RUN_TS}/`

In [ ]:
# --- Storage configuration ---
# Save outputs (CSVs, replays, JSON bundle) to Google Drive so they survive
# a Colab runtime disconnect. Set False to use local/Colab filesystem.
USE_GOOGLE_DRIVE = True

# Path inside Google Drive (relative to MyDrive/) when USE_GOOGLE_DRIVE is True.
DRIVE_BASE_DIR = "reinforce-tactics/balance_analysis"

# --- Mount Google Drive if enabled ---
if USE_GOOGLE_DRIVE:
    try:
        from google.colab import drive

        drive.mount("/content/drive")
        SAVE_DIR = Path("/content/drive/MyDrive") / DRIVE_BASE_DIR
        SAVE_DIR.mkdir(parents=True, exist_ok=True)
        print("Google Drive mounted successfully.")
        print(f"Save directory: {SAVE_DIR}")
    except ImportError:
        print("WARNING: google.colab not available (not running in Colab).")
        print("  Falling back to local storage.")
        USE_GOOGLE_DRIVE = False
        SAVE_DIR = Path("tournament_results")
    except Exception as e:
        print(f"WARNING: Failed to mount Google Drive: {e}")
        print("  Falling back to local storage.")
        USE_GOOGLE_DRIVE = False
        SAVE_DIR = Path("tournament_results")
else:
    SAVE_DIR = Path("tournament_results")
    print(f"Using local storage: {SAVE_DIR}")
    print("  Tip: Set USE_GOOGLE_DRIVE = True to persist results to Google Drive.")

## Environment Overrides

Knobs for testing balance changes without editing source. Edit `ENV_OVERRIDES` below, then re-run from this cell downward. Each run snapshots the override config alongside the CSVs so you can compare runs.

Use `RUN_TAG` to label what you're testing (e.g. `"baseline"`, `"warrior_225"`, `"p2_plus_50"`, `"p2_starts_warrior"`) — it gets baked into the output directory name.

**Available knobs:**
- `starting_gold[p]` — per-player starting gold. Default 250 each. Try `{1: 250, 2: 300}` for a komi-style P2 offset.
- `starting_unit[p]` — spawn a free unit on player `p`'s HQ at game start (no gold cost, doesn't count as a turn-1 build). Empty/`None` = no unit. Example: `{2: "W"}` gives P2 a free warrior on their HQ to offset turn-order disadvantage.
- `unit_cost[unit_letter]` — override unit gold cost. Example: `{"W": 225}` bumps warrior cost.
- `unit_stat[unit_letter]` — override stats (`health`, `attack`, `defence`, `movement`). Example: `{"W": {"health": 12}}`.
- `income[structure]` — override per-turn income. Example: `{"headquarters": 130}` to slow economy.

Unit letters: `W` Warrior, `A` Archer, `C` Cleric, `K` Knight, `R` Rogue, `M` Mage, `S` Sorcerer, `B` Barbarian.

In [ ]:
# Tag this run so output is identifiable. Change when testing variants.
RUN_TAG = "baseline"

# Edit this dict to test balance changes. Empty dicts = no override.
ENV_OVERRIDES = {
    "starting_gold": {1: 250, 2: 250},  # per-player. e.g. {1: 250, 2: 300} for P2 komi
    "starting_unit": {},  # e.g. {2: "W"} - free unit on P2's HQ at game start
    "unit_cost": {},  # e.g. {"W": 225, "A": 225}
    "unit_stat": {},  # e.g. {"W": {"health": 12, "attack": 8}}
    "income": {},  # keys: "headquarters", "building", "tower"
}


def apply_env_overrides(cfg):
    """Mutate game constants and patch GameState init for per-player gold and starting units.

    Idempotent: re-runs replace prior overrides. Originals are cached on first
    call so re-running with empty overrides restores defaults.
    """
    import reinforcetactics.constants as C
    import reinforcetactics.game.mechanics as mech_mod
    from reinforcetactics.core.game_state import GameState
    from reinforcetactics.core.unit import Unit

    if not hasattr(apply_env_overrides, "_originals"):
        apply_env_overrides._originals = {
            "unit_data": {ut: dict(C.UNIT_DATA[ut]) for ut in C.UNIT_DATA},
            "income": {
                "headquarters": C.HEADQUARTERS_INCOME,
                "building": C.BUILDING_INCOME,
                "tower": C.TOWER_INCOME,
            },
            "starting_gold": C.STARTING_GOLD,
            "gs_init": GameState.__init__,
        }
    orig = apply_env_overrides._originals

    # 1. UNIT_DATA - restore then apply (mutate in place so all importers see it)
    for ut, base in orig["unit_data"].items():
        C.UNIT_DATA[ut].clear()
        C.UNIT_DATA[ut].update(base)
    for ut, cost in cfg.get("unit_cost", {}).items():
        if ut not in C.UNIT_DATA:
            raise KeyError(f"Unknown unit type {ut!r}")
        C.UNIT_DATA[ut]["cost"] = cost
    for ut, stats in cfg.get("unit_stat", {}).items():
        if ut not in C.UNIT_DATA:
            raise KeyError(f"Unknown unit type {ut!r}")
        C.UNIT_DATA[ut].update(stats)

    # 2. Income - patch both modules since mechanics imports the names directly
    inc = cfg.get("income", {})
    for key, attr in [("headquarters", "HEADQUARTERS_INCOME"), ("building", "BUILDING_INCOME"), ("tower", "TOWER_INCOME")]:
        val = inc.get(key, orig["income"][key])
        setattr(C, attr, val)
        setattr(mech_mod, attr, val)

    # 3. Per-player starting gold + starting unit via init wrapper
    gold_map = {p: g for p, g in cfg.get("starting_gold", {}).items() if g is not None}
    unit_map = {p: u for p, u in cfg.get("starting_unit", {}).items() if u}  # drop blanks
    for p, ut in unit_map.items():
        if ut not in C.UNIT_DATA:
            raise KeyError(f"Unknown unit type {ut!r} in starting_unit")

    GameState.__init__ = orig["gs_init"]  # always restore first
    if gold_map or unit_map:
        base_init = orig["gs_init"]

        def find_hq(grid, player):
            for row in grid.tiles:
                for tile in row:
                    if getattr(tile, "type", None) == "h" and getattr(tile, "player", None) == player:
                        return tile.x, tile.y
            return None

        def patched_init(self, *args, **kwargs):
            base_init(self, *args, **kwargs)
            for p, g in gold_map.items():
                if p in self.player_gold:
                    self.player_gold[p] = g
            for p, ut in unit_map.items():
                if p not in self.player_gold:
                    continue
                pos = find_hq(self.grid, p)
                if pos is None:
                    continue
                x, y = pos
                if self.get_unit_at_position(x, y) is not None:
                    continue
                self.units.append(Unit(ut, x, y, p))
                self._invalidate_cache()
                # Record so it shows up in replay analysis
                self.record_action("create_unit", unit_type=ut, x=x, y=y, player=p)

        GameState.__init__ = patched_init

    print("Effective overrides:")
    default_gold = orig["starting_gold"]
    print(f"  starting_gold: {gold_map or f'default ({default_gold} both)'}")
    print(f"  starting_unit: {unit_map or 'none'}")
    print(f"  unit costs:    {[(u, C.UNIT_DATA[u]['cost']) for u in C.UNIT_DATA]}")
    print(f"  income:        HQ={C.HEADQUARTERS_INCOME} Bldg={C.BUILDING_INCOME} Twr={C.TOWER_INCOME}")
    if cfg.get("unit_stat"):
        print(f"  unit stat overrides: {cfg['unit_stat']}")


apply_env_overrides(ENV_OVERRIDES)

OUTPUT_DIR = str(SAVE_DIR / f"balance_analysis_{RUN_TAG}_{RUN_TS}")
print("Output ->", OUTPUT_DIR)

## Run Tournament

In [ ]:
import json
import logging
import os

from reinforcetactics.tournament import TournamentConfig, TournamentRunner
from reinforcetactics.tournament.bots import discover_all_bots, discover_builtin_bots

logging.basicConfig(level=logging.WARNING, format="%(asctime)s %(levelname)s %(message)s")

if USE_BUILTIN_ONLY:
    bots = discover_builtin_bots()
else:
    bots = discover_all_bots(models_dir="models", include_llm=True, include_models=True)

print(f"Bots: {[b.name for b in bots]}")

config = TournamentConfig(
    name=f"Balance Analysis - {RUN_TAG}",
    maps=MAPS,
    map_pool_mode="all",
    games_per_side=GAMES_PER_SIDE,
    max_turns=MAX_TURNS,
    output_dir=OUTPUT_DIR,
    save_replays=True,
    concurrent_games=CONCURRENT,
)

# Snapshot the override config alongside the run for traceability
os.makedirs(OUTPUT_DIR, exist_ok=True)
with open(Path(OUTPUT_DIR) / "env_overrides.json", "w") as f:
    json.dump(
        {
            "run_tag": RUN_TAG,
            "run_ts": RUN_TS,
            "env_overrides": ENV_OVERRIDES,
            "games_per_side": GAMES_PER_SIDE,
            "max_turns": MAX_TURNS,
            "maps": MAPS,
        },
        f,
        indent=2,
    )

runner = TournamentRunner(config)
results = runner.run(bots)
runner.export_results()
print(f"\nTournament complete. {len(results.game_results)} games played.")

## First-Turn Advantage

For each game, `winner == 1` means whoever was assigned the P1 slot won, `winner == 2` means P2 won, `winner == 0` is a draw. Because the tournament swaps sides, we get a clean P1-vs-P2 measurement that's independent of which bot is stronger.

Three views per map:
- **P1 win rate (strict)** — wins only, draws excluded from numerator
- **P1 score** — wins count 1, draws count 0.5 (komi-style)
- **Draw rate** — high draw rates mask the signal, so reported explicitly

Wilson 95% confidence intervals included so you can tell whether observed asymmetries are real. Also aggregated by **map size bucket** (small / medium / large) — that's the headline view for whether the warrior-rush concern is a small-map issue.

In [ ]:
import math

import pandas as pd


def wilson_ci(k, n, z=1.96):
    if n == 0:
        return (float("nan"), float("nan"), float("nan"))
    p = k / n
    denom = 1 + z * z / n
    centre = (p + z * z / (2 * n)) / denom
    margin = z * math.sqrt(p * (1 - p) / n + z * z / (4 * n * n)) / denom
    return (p, max(0.0, centre - margin), min(1.0, centre + margin))


games = results.game_results
games_df = pd.DataFrame([g.to_dict() for g in games])
games_df["map_stem"] = games_df["map"].apply(lambda m: Path(m).stem)
games_df["size"] = games_df["map_stem"].map(lambda s: f"{MAP_SIZE.get(s, ('?', '?'))[0]}x{MAP_SIZE.get(s, ('?', '?'))[1]}")
games_df["bucket"] = games_df["map_stem"].map(MAP_BUCKET).fillna("unknown")
games_df.head()

In [ ]:
def _summary_row(label, df, extra=None):
    n = len(df)
    p1w = int((df["winner"] == 1).sum())
    p2w = int((df["winner"] == 2).sum())
    dw = int((df["winner"] == 0).sum())
    decisive = p1w + p2w
    p1_strict, lo_s, hi_s = wilson_ci(p1w, decisive)
    score_num = p1w + 0.5 * dw
    p1_score, lo_sc, hi_sc = wilson_ci(int(round(score_num * 2)), n * 2)
    row = dict(
        label=label,
        n=n,
        p1_wins=p1w,
        p2_wins=p2w,
        draws=dw,
        p1_winrate_strict=round(p1_strict, 3) if decisive else None,
        ci95_strict=f"[{lo_s:.2f}, {hi_s:.2f}]" if decisive else "",
        p1_score=round(p1_score, 3),
        ci95_score=f"[{lo_sc:.2f}, {hi_sc:.2f}]",
        draw_rate=round(dw / n, 3) if n else None,
    )
    if extra:
        row.update(extra)
    return row


# Per-map summary
rows = [
    _summary_row(
        f"{stem} ({MAP_BUCKET.get(stem, '?')}, {MAP_SIZE.get(stem, ('?', '?'))[0]}x{MAP_SIZE.get(stem, ('?', '?'))[1]})",
        g,
        extra={"map": stem, "bucket": MAP_BUCKET.get(stem)},
    )
    for stem, g in games_df.groupby("map_stem")
]
# Per-bucket summary - the headline "does it scale with map size?" view
rows += [_summary_row(f"BUCKET: {b}", g, extra={"map": f"BUCKET_{b}", "bucket": b}) for b, g in games_df.groupby("bucket")]
# Overall
rows.append(_summary_row("ALL", games_df, extra={"map": "ALL", "bucket": "all"}))

p1_by_map = pd.DataFrame(rows)
p1_by_map

### Per-bot first-player advantage

Pair-level breakdown: for each (botA, botB) matchup, what happens when botA plays P1 vs when botB plays P1? A bot with a strong P1 advantage should win more often when playing P1 than when playing P2. This isolates whether the advantage is structural (turn order) or just bot-strength asymmetry.

In [ ]:
def per_bot_p1_table(df):
    rows = []
    pairs = set()
    for _, r in df.iterrows():
        a, b = r["bot1"], r["bot2"]
        pairs.add(tuple(sorted([a, b])))
    for a, b in sorted(pairs):
        sub = df[((df["bot1"] == a) & (df["bot2"] == b)) | ((df["bot1"] == b) & (df["bot2"] == a))]
        a_as_p1 = sub[sub["bot1"] == a]
        b_as_p1 = sub[sub["bot1"] == b]
        a_p1_n = len(a_as_p1)
        b_p1_n = len(b_as_p1)
        rows.append(
            dict(
                bot_a=a,
                bot_b=b,
                a_winrate_as_p1=round((a_as_p1["winner"] == 1).sum() / a_p1_n, 3) if a_p1_n else None,
                a_winrate_as_p2=round((b_as_p1["winner"] == 2).sum() / b_p1_n, 3) if b_p1_n else None,
                b_winrate_as_p1=round((b_as_p1["winner"] == 1).sum() / b_p1_n, 3) if b_p1_n else None,
                b_winrate_as_p2=round((a_as_p1["winner"] == 2).sum() / a_p1_n, 3) if a_p1_n else None,
                n_games=len(sub),
            )
        )
    return pd.DataFrame(rows)


per_bot_p1_table(games_df)

## Bot-level Performance

How does each bot do overall, and does the result hold across skill levels? Three views:

1. **Standings** — wins / losses / draws / winrate / Elo per bot.
2. **Per-bot win rate by map size bucket** — does a bot's edge widen on bigger maps? Does the warrior-rush hypothesis hold across all skill levels?
3. **Per-bot unit composition** — does a higher-skill bot diversify away from warriors? If only SimpleBot spams warriors and AdvancedBot doesn't, the issue is more bot-strategy than game balance.

In [ ]:
# 1. Standings
standings = results.get_standings()
standings_rows = [s.to_dict() for s in standings]
standings_df = pd.DataFrame([{k: v for k, v in r.items() if k != "per_map_stats"} for r in standings_rows]).sort_values(
    "win_rate", ascending=False
)
print("=== Standings ===")
print(standings_df.to_string(index=False))


# 2. Per-bot win rate by map size bucket
def per_bot_bucket(df):
    rows = []
    bots_seen = set(df["bot1"]) | set(df["bot2"])
    for bot in sorted(bots_seen):
        for bucket, gb in df.groupby("bucket"):
            played = gb[(gb["bot1"] == bot) | (gb["bot2"] == bot)]
            n = len(played)
            if n == 0:
                continue
            wins = ((played["bot1"] == bot) & (played["winner"] == 1)).sum() + (
                (played["bot2"] == bot) & (played["winner"] == 2)
            ).sum()
            losses = ((played["bot1"] == bot) & (played["winner"] == 2)).sum() + (
                (played["bot2"] == bot) & (played["winner"] == 1)
            ).sum()
            draws = (played["winner"] == 0).sum()
            wr, lo, hi = wilson_ci(int(wins), n)
            rows.append(
                dict(
                    bot=bot,
                    bucket=bucket,
                    n=n,
                    wins=int(wins),
                    losses=int(losses),
                    draws=int(draws),
                    winrate=round(wr, 3),
                    ci95=f"[{lo:.2f}, {hi:.2f}]",
                )
            )
    return pd.DataFrame(rows).sort_values(["bot", "bucket"])


bot_bucket_df = per_bot_bucket(games_df)
print("\n=== Per-bot winrate by map bucket ===")
print(bot_bucket_df.to_string(index=False))
bot_bucket_df

## Unit Competitiveness

Walk every replay's `actions` list, count `create_unit` events per (player, unit_type, game), then aggregate.

Three views:

1. **Production share** — what fraction of all units built across the tournament is each unit type? Tells you what bots actually choose to build.
2. **Gold share** — same but weighted by unit cost. Better signal for "where are players investing?"
3. **Win correlation** — for each unit, what's the gold-share-on-this-unit in winning vs losing games? A wide gap means building more (or less) of that unit predicts winning.

In [ ]:
import glob
import json

from reinforcetactics.constants import UNIT_DATA

UNIT_COSTS = {ut: UNIT_DATA[ut]["cost"] for ut in UNIT_DATA}
UNIT_NAMES = {ut: UNIT_DATA[ut].get("name", ut) for ut in UNIT_DATA}

# Known bot names from the standings; used to disambiguate filenames where
# the map_stem contains underscores (difficult_terrain, corner_points, etc).
KNOWN_BOTS = {s.bot_name for s in standings}
# Known map stems from the configured pool; second fallback when neither
# game_info.map_file nor the replay's parent directory yields a usable stem.
KNOWN_MAP_STEMS = set(MAP_SIZE.keys())

replay_files = sorted(glob.glob(f"{OUTPUT_DIR}/replays/**/*.json", recursive=True))
print(f"Found {len(replay_files)} replays")


def _strip_map_suffix(text, map_stem):
    """Strip a trailing ``_<map_stem>`` from a filename component."""
    suffix = f"_{map_stem}"
    if map_stem and text.endswith(suffix):
        return text[: -len(suffix)]
    return text


def _resolve_map_stem(path, info):
    """Find the map_stem for a replay using three fallback sources.

    1. ``game_info.map_file`` (the original location used by the runner).
       Older replays don't populate this field.
    2. The immediate parent folder of the replay file. The runner writes
       replays into ``<output_dir>/replays/<map_stem>/<file>.json``.
    3. Suffix-match against the configured ``MAP_POOL`` map stems (handles
       multi-word names like ``corner_points``).
    """
    map_file = info.get("map_file") or ""
    if map_file:
        return Path(map_file).stem
    parent = Path(path).parent.name
    if parent and parent != "replays":
        return parent
    fname = Path(path).stem
    for stem in sorted(KNOWN_MAP_STEMS, key=len, reverse=True):
        if fname.endswith(f"_{stem}"):
            return stem
    return "unknown"


def parse_replay(path):
    with open(path) as f:
        d = json.load(f)
    info = d.get("game_info", {})
    actions = d.get("actions", [])
    winner = info.get("winner", 0)
    map_stem = _resolve_map_stem(path, info)
    fname = Path(path).stem

    bot1 = bot2 = None
    if "_vs_" in fname:
        left, right = fname.split("_vs_", 1)
        # Right side is "<bot2>_<map_stem>" or "<bot2>_<P1wins|P2wins|Draw>".
        # First try stripping the known map_stem (handles multi-word maps);
        # then fall back to rsplit for legacy result-suffix filenames.
        right_stripped = _strip_map_suffix(right, map_stem)
        bot2 = right_stripped if right_stripped != right else right.rsplit("_", 1)[0]
        # Left side is "game_<TS>_<bot1>" with TS like "20260510_150447" containing one underscore.
        # Walk back from the right until we find a known-bot suffix, otherwise take the trailing token.
        bot1 = None
        for k in range(1, 5):
            candidate = "_".join(left.rsplit("_", k)[-k:])
            if candidate in KNOWN_BOTS:
                bot1 = candidate
                break
        if bot1 is None:
            bot1 = left.split("_", 2)[-1] if left.count("_") >= 2 else left

    units_by_player = {1: [], 2: []}
    for a in actions:
        if a.get("type") == "create_unit":
            p = a.get("player")
            ut = a.get("unit_type")
            if p in units_by_player and ut in UNIT_COSTS:
                units_by_player[p].append(ut)
    return dict(
        path=path,
        map=map_stem,
        winner=winner,
        bot1=bot1,
        bot2=bot2,
        p1_units=units_by_player[1],
        p2_units=units_by_player[2],
        total_turns=info.get("total_turns"),
    )


parsed = [parse_replay(p) for p in replay_files]
print(f"Parsed {len(parsed)} replays")
unknown_pairs = [(r["bot1"], r["bot2"]) for r in parsed if r["bot1"] not in KNOWN_BOTS or r["bot2"] not in KNOWN_BOTS]
if unknown_pairs:
    print(f"WARNING: {len(unknown_pairs)} replays have unrecognised bot names: {set(unknown_pairs)}")
unknown_maps = [r["map"] for r in parsed if r["map"] == "unknown"]
if unknown_maps:
    print(f"WARNING: {len(unknown_maps)} replays have unknown map_stem")
parsed[0] if parsed else None

In [ ]:
from collections import Counter

production = Counter()
gold_spent = Counter()
games_with_unit = Counter()

for r in parsed:
    for player in (1, 2):
        units = r[f"p{player}_units"]
        seen = set()
        for ut in units:
            production[ut] += 1
            gold_spent[ut] += UNIT_COSTS[ut]
            seen.add(ut)
        for ut in seen:
            games_with_unit[ut] += 1

total_units = sum(production.values()) or 1
total_gold = sum(gold_spent.values()) or 1
total_player_games = 2 * len(parsed) or 1

prod_rows = []
for ut in sorted(UNIT_COSTS, key=lambda u: -production[u]):
    prod_rows.append(
        dict(
            unit=ut,
            name=UNIT_NAMES[ut],
            cost=UNIT_COSTS[ut],
            produced=production[ut],
            prod_share=round(production[ut] / total_units, 3),
            gold_spent=gold_spent[ut],
            gold_share=round(gold_spent[ut] / total_gold, 3),
            pct_player_games_built=round(games_with_unit[ut] / total_player_games, 3),
        )
    )
production_df = pd.DataFrame(prod_rows)
production_df

In [ ]:
records = []
for r in parsed:
    for player in (1, 2):
        units = r[f"p{player}_units"]
        if r["winner"] == 0:
            outcome = "draw"
        elif r["winner"] == player:
            outcome = "win"
        else:
            outcome = "loss"
        gold_total = sum(UNIT_COSTS[u] for u in units) or 0
        row = dict(map=r["map"], player=player, outcome=outcome, gold_total=gold_total, n_units=len(units))
        for ut in UNIT_COSTS:
            row[f"gold_{ut}"] = sum(UNIT_COSTS[u] for u in units if u == ut)
            row[f"share_{ut}"] = (row[f"gold_{ut}"] / gold_total) if gold_total else 0.0
        records.append(row)

pg = pd.DataFrame(records)
pg["bucket"] = pg["map"].map(MAP_BUCKET).fillna("unknown")
print(f"Player-games: {len(pg)} (win/loss/draw: {pg['outcome'].value_counts().to_dict()})")

diff_rows = []
for ut in UNIT_COSTS:
    win_share = pg[pg["outcome"] == "win"][f"share_{ut}"]
    loss_share = pg[pg["outcome"] == "loss"][f"share_{ut}"]
    if len(win_share) == 0 or len(loss_share) == 0:
        continue
    diff_rows.append(
        dict(
            unit=ut,
            name=UNIT_NAMES[ut],
            gold_share_win=round(win_share.mean(), 3),
            gold_share_loss=round(loss_share.mean(), 3),
            delta=round(win_share.mean() - loss_share.mean(), 3),
            n_win_games=len(win_share),
            n_loss_games=len(loss_share),
        )
    )
unit_outcome_df = pd.DataFrame(diff_rows).sort_values("delta", ascending=False)
unit_outcome_df

### Per-bot unit composition

If only `SimpleBot` spams warriors but `AdvancedBot` builds a balanced mix, warrior dominance may be a bot-skill artifact rather than a balance issue. If *every* bot funnels gold into warriors, that's a real balance signal.

In [ ]:
# Per-bot unit composition: aggregate gold-share-per-unit across each bot's player-games.
# Bot identity comes from the replay filename (parsed in unit-parse cell).
bot_unit_records = []
for r in parsed:
    if not (r.get("bot1") and r.get("bot2")):
        continue
    for player, bot_name in [(1, r["bot1"]), (2, r["bot2"])]:
        units = r[f"p{player}_units"]
        gold_total = sum(UNIT_COSTS[u] for u in units) or 0
        outcome = "draw" if r["winner"] == 0 else ("win" if r["winner"] == player else "loss")
        row = dict(
            bot=bot_name,
            map=r["map"],
            bucket=MAP_BUCKET.get(r["map"], "unknown"),
            outcome=outcome,
            gold_total=gold_total,
            n_units=len(units),
        )
        for ut in UNIT_COSTS:
            row[f"gold_{ut}"] = sum(UNIT_COSTS[u] for u in units if u == ut)
            row[f"share_{ut}"] = (row[f"gold_{ut}"] / gold_total) if gold_total else 0.0
        bot_unit_records.append(row)

bot_pg = pd.DataFrame(bot_unit_records)

# Aggregate gold share per unit per bot
bot_unit_rows = []
for bot, g in bot_pg.groupby("bot"):
    win_g = g[g["outcome"] == "win"]
    for ut in UNIT_COSTS:
        bot_unit_rows.append(
            dict(
                bot=bot,
                unit=ut,
                name=UNIT_NAMES[ut],
                avg_gold_share=round(g[f"share_{ut}"].mean(), 3),
                win_gold_share=round(win_g[f"share_{ut}"].mean(), 3) if len(win_g) else None,
                avg_units_built=round(g[f"gold_{ut}"].mean() / UNIT_COSTS[ut], 2),
                n_player_games=len(g),
            )
        )
bot_unit_df = pd.DataFrame(bot_unit_rows).sort_values(["bot", "avg_gold_share"], ascending=[True, False])
print("=== Per-bot unit gold share ===")
print(bot_unit_df.to_string(index=False))

# Per-bot per-bucket: does bot X build differently on small vs large maps?
bot_bucket_unit_rows = []
for (bot, bucket), g in bot_pg.groupby(["bot", "bucket"]):
    for ut in UNIT_COSTS:
        bot_bucket_unit_rows.append(
            dict(
                bot=bot,
                bucket=bucket,
                unit=ut,
                name=UNIT_NAMES[ut],
                avg_gold_share=round(g[f"share_{ut}"].mean(), 3),
                n_player_games=len(g),
            )
        )
bot_bucket_unit_df = pd.DataFrame(bot_bucket_unit_rows).sort_values(
    ["bot", "bucket", "avg_gold_share"], ascending=[True, True, False]
)
bot_bucket_unit_df

### Unit composition by map size and individual map

Does warrior dominance scale with map size? The bucket view (small/medium/large) gives the headline answer; per-map view shows whether specific maps are outliers.

In [ ]:
map_unit_rows = []
for map_stem, g in pg.groupby("map"):
    for ut in UNIT_COSTS:
        share = g[f"share_{ut}"].mean()
        win_share = g[g["outcome"] == "win"][f"share_{ut}"].mean() if (g["outcome"] == "win").any() else float("nan")
        map_unit_rows.append(
            dict(
                map=map_stem,
                bucket=MAP_BUCKET.get(map_stem, "?"),
                unit=ut,
                name=UNIT_NAMES[ut],
                avg_gold_share=round(share, 3),
                win_gold_share=round(win_share, 3) if not pd.isna(win_share) else None,
                n_player_games=len(g),
            )
        )
map_unit_df = pd.DataFrame(map_unit_rows).sort_values(["bucket", "map", "avg_gold_share"], ascending=[True, True, False])

bucket_unit_rows = []
for bucket, g in pg.groupby("bucket"):
    for ut in UNIT_COSTS:
        share = g[f"share_{ut}"].mean()
        win_share = g[g["outcome"] == "win"][f"share_{ut}"].mean() if (g["outcome"] == "win").any() else float("nan")
        loss_share = g[g["outcome"] == "loss"][f"share_{ut}"].mean() if (g["outcome"] == "loss").any() else float("nan")
        bucket_unit_rows.append(
            dict(
                bucket=bucket,
                unit=ut,
                name=UNIT_NAMES[ut],
                avg_gold_share=round(share, 3),
                win_gold_share=round(win_share, 3) if not pd.isna(win_share) else None,
                loss_gold_share=round(loss_share, 3) if not pd.isna(loss_share) else None,
                n_player_games=len(g),
            )
        )
bucket_unit_df = pd.DataFrame(bucket_unit_rows).sort_values(["bucket", "avg_gold_share"], ascending=[True, False])
print("=== By bucket (small / medium / large) ===")
print(bucket_unit_df.to_string(index=False))
print("\n=== By individual map ===")
map_unit_df

## Support Unit Impact (Tier 1)

Damage and gold-share understate non-damaging units (Cleric, Sorcerer) because their value is indirect: keeping a frontline unit alive, multiplying an ally's damage, or granting an extra action.

This section walks the replay action stream and counts:

- **Cleric**: total HP healed, heal action count, cure action count
- **Sorcerer**: haste / attack-buff / defence-buff cast counts

Aggregated per (player, game), then per bot. Also reports a **win-correlation view**: when a player built ≥1 cleric (or sorcerer), did they win more often than when they didn't?

Per-unit-instance attribution (assists, damage prevented) requires unit tracking through moves and is Tier 2 — kept out for now to keep this purely replay-driven.

In [ ]:
from collections import defaultdict

# Walk every replay's action stream and aggregate support activity per (player, game).
# We re-load the JSON because parse_replay() only retained create_unit events.
SUPPORT_ACTIONS = ("heal", "cure", "haste", "attack_buff", "defence_buff")

support_records = []
for r in parsed:
    with open(r["path"]) as f:
        d = json.load(f)
    actions = d.get("actions", [])

    counters = {1: defaultdict(int), 2: defaultdict(int)}
    for a in actions:
        atype = a.get("type")
        player = a.get("player")
        if player not in (1, 2):
            continue
        if atype == "create_unit":
            ut = a.get("unit_type")
            if ut == "C":
                counters[player]["cleric_count"] += 1
            elif ut == "S":
                counters[player]["sorcerer_count"] += 1
        elif atype == "heal":
            counters[player]["heal_hp"] += a.get("amount", 0)
            counters[player]["heal_actions"] += 1
        elif atype == "cure":
            counters[player]["cure_actions"] += 1
        elif atype == "haste":
            counters[player]["haste_actions"] += 1
        elif atype == "attack_buff":
            counters[player]["attack_buff_actions"] += 1
        elif atype == "defence_buff":
            counters[player]["defence_buff_actions"] += 1

    for player in (1, 2):
        c = counters[player]
        bot_name = r["bot1"] if player == 1 else r["bot2"]
        outcome = "draw" if r["winner"] == 0 else ("win" if r["winner"] == player else "loss")
        nc = c["cleric_count"]
        ns = c["sorcerer_count"]
        buffs = c["haste_actions"] + c["attack_buff_actions"] + c["defence_buff_actions"]
        support_records.append(
            dict(
                map=r["map"],
                bucket=MAP_BUCKET.get(r["map"], "unknown"),
                player=player,
                bot=bot_name,
                outcome=outcome,
                cleric_count=nc,
                sorcerer_count=ns,
                heal_hp=c["heal_hp"],
                heal_actions=c["heal_actions"],
                cure_actions=c["cure_actions"],
                haste_actions=c["haste_actions"],
                attack_buff_actions=c["attack_buff_actions"],
                defence_buff_actions=c["defence_buff_actions"],
                buffs_cast=buffs,
                heal_hp_per_cleric=round(c["heal_hp"] / nc, 2) if nc else 0,
                buffs_per_sorcerer=round(buffs / ns, 2) if ns else 0,
            )
        )

support_df = pd.DataFrame(support_records)
n_with_cleric = int((support_df["cleric_count"] > 0).sum())
n_with_sorcerer = int((support_df["sorcerer_count"] > 0).sum())
print(f"Player-games with >=1 Cleric:   {n_with_cleric} / {len(support_df)}")
print(f"Player-games with >=1 Sorcerer: {n_with_sorcerer} / {len(support_df)}")
print(f"Total HP healed: {int(support_df['heal_hp'].sum())}")
print(f"Total buffs cast: {int(support_df['buffs_cast'].sum())}")
support_df.head()

In [ ]:
# Per-bot averages. "avg_*_per_built" is conditioned on games where the support
# unit was actually built; raw averages would be drowned out by the >85% of
# games with zero support units.
support_per_bot_rows = []
for bot, g in support_df.groupby("bot"):
    cleric_games = g[g["cleric_count"] > 0]
    sorcerer_games = g[g["sorcerer_count"] > 0]
    support_per_bot_rows.append(
        dict(
            bot=bot,
            n_games=len(g),
            n_with_cleric=len(cleric_games),
            avg_clerics_built=round(g["cleric_count"].mean(), 2),
            avg_heal_hp_when_built=round(cleric_games["heal_hp"].mean(), 1) if len(cleric_games) else 0,
            avg_heal_per_cleric=round(cleric_games["heal_hp_per_cleric"].mean(), 2) if len(cleric_games) else 0,
            avg_cures_when_built=round(cleric_games["cure_actions"].mean(), 2) if len(cleric_games) else 0,
            n_with_sorcerer=len(sorcerer_games),
            avg_sorcerers_built=round(g["sorcerer_count"].mean(), 2),
            avg_buffs_cast_when_built=round(sorcerer_games["buffs_cast"].mean(), 1) if len(sorcerer_games) else 0,
            avg_buffs_per_sorcerer=round(sorcerer_games["buffs_per_sorcerer"].mean(), 2) if len(sorcerer_games) else 0,
        )
    )
support_per_bot_df = pd.DataFrame(support_per_bot_rows)
print("=== Per-bot support unit activity ===")
print(support_per_bot_df.to_string(index=False))
support_per_bot_df

In [ ]:
# Win-correlation: when a player built >=1 cleric (or sorcerer), did they win more often
# than when they didn't? Wilson 95% CI for both halves.
support_winrate_rows = []
for unit_label, mask_col in [("Cleric", "cleric_count"), ("Sorcerer", "sorcerer_count")]:
    with_unit = support_df[support_df[mask_col] > 0]
    without_unit = support_df[support_df[mask_col] == 0]

    def _wr(df):
        if len(df) == 0:
            return None, None
        wins = int((df["outcome"] == "win").sum())
        wr, lo, hi = wilson_ci(wins, len(df))
        return round(wr, 3), f"[{lo:.2f}, {hi:.2f}]"

    wr_with, ci_with = _wr(with_unit)
    wr_without, ci_without = _wr(without_unit)
    delta = round((wr_with or 0) - (wr_without or 0), 3) if wr_with is not None and wr_without is not None else None
    support_winrate_rows.append(
        dict(
            unit=unit_label,
            n_with=len(with_unit),
            winrate_with=wr_with,
            ci95_with=ci_with,
            n_without=len(without_unit),
            winrate_without=wr_without,
            ci95_without=ci_without,
            delta=delta,
        )
    )
support_winrate_df = pd.DataFrame(support_winrate_rows)
print("=== Win-correlation: built vs not built ===")
print(support_winrate_df.to_string(index=False))
support_winrate_df

## Export

Writes CSVs into the run output directory (alongside the `env_overrides.json` snapshot, so each run is fully reproducible):

- `p1_advantage_by_map.csv` — first-turn advantage per map and per size bucket
- `unit_production_summary.csv` — overall production / gold share / build rate per unit
- `unit_gold_share_by_outcome.csv` — gold-share-on-unit in winning vs losing player-games
- `unit_gold_share_by_map.csv` / `_by_bucket.csv` — same, split per map / size bucket
- `bot_standings.csv` — per-bot wins/losses/draws/winrate/Elo
- `bot_winrate_by_bucket.csv` — per-bot winrate split by map size
- `bot_unit_gold_share.csv` / `_by_bucket.csv` — per-bot unit composition (does each skill level build the same way?)
- `support_unit_impact.csv` — per-(player, game) heal HP, cures, and buffs cast (Tier 1 support metrics)
- `support_unit_impact_by_bot.csv` — same, averaged per bot (conditioned on games where the support unit was built)
- `support_unit_winrate.csv` — winrate when ≥1 cleric/sorcerer was built vs not

In [ ]:
out = Path(OUTPUT_DIR)
p1_by_map.to_csv(out / "p1_advantage_by_map.csv", index=False)
production_df.to_csv(out / "unit_production_summary.csv", index=False)
unit_outcome_df.to_csv(out / "unit_gold_share_by_outcome.csv", index=False)
map_unit_df.to_csv(out / "unit_gold_share_by_map.csv", index=False)
bucket_unit_df.to_csv(out / "unit_gold_share_by_bucket.csv", index=False)
standings_df.to_csv(out / "bot_standings.csv", index=False)
bot_bucket_df.to_csv(out / "bot_winrate_by_bucket.csv", index=False)
bot_unit_df.to_csv(out / "bot_unit_gold_share.csv", index=False)
bot_bucket_unit_df.to_csv(out / "bot_unit_gold_share_by_bucket.csv", index=False)
support_df.to_csv(out / "support_unit_impact.csv", index=False)
support_per_bot_df.to_csv(out / "support_unit_impact_by_bot.csv", index=False)
support_winrate_df.to_csv(out / "support_unit_winrate.csv", index=False)
print("Wrote CSVs to", out)
for p in sorted(out.glob("*.csv")):
    print(" ", p)
print("\nRun config snapshot:", out / "env_overrides.json")

### Raw data and offline bundle

In addition to the summary CSVs, save the row-level data so you can reanalyse offline without re-running the tournament:

- `games_raw.csv` — one row per game (bot1, bot2, winner, turns, map, size, bucket)
- `player_game_units.csv` — one row per player-game with the gold-share-per-unit columns (the source data behind the bucket/map summaries)
- `analysis_bundle.json` — single self-contained JSON with run metadata, override snapshot, all summary tables, and parsed replays. Easiest to load from another notebook or script.

In [ ]:
import shutil

# Raw row-level data
games_df.to_csv(out / "games_raw.csv", index=False)
pg.to_csv(out / "player_game_units.csv", index=False)
bot_pg.to_csv(out / "bot_player_game_units.csv", index=False)

# Parsed replay digests (small enough to bundle - units only, not full action history)
parsed_digest = [
    dict(
        path=str(Path(r["path"]).name),
        map=r["map"],
        bucket=MAP_BUCKET.get(r["map"], "unknown"),
        winner=r["winner"],
        bot1=r["bot1"],
        bot2=r["bot2"],
        total_turns=r["total_turns"],
        p1_units=r["p1_units"],
        p2_units=r["p2_units"],
    )
    for r in parsed
]

# Combined offline-analysis bundle
bundle = {
    "run_tag": RUN_TAG,
    "run_ts": RUN_TS,
    "env_overrides": ENV_OVERRIDES,
    "games_per_side": GAMES_PER_SIDE,
    "max_turns": MAX_TURNS,
    "maps": MAPS,
    "map_size": {k: list(v) for k, v in MAP_SIZE.items()},
    "map_bucket": MAP_BUCKET,
    "unit_costs": UNIT_COSTS,
    "summaries": {
        "p1_by_map": p1_by_map.to_dict(orient="records"),
        "production": production_df.to_dict(orient="records"),
        "unit_outcome": unit_outcome_df.to_dict(orient="records"),
        "map_unit": map_unit_df.to_dict(orient="records"),
        "bucket_unit": bucket_unit_df.to_dict(orient="records"),
        "bot_standings": standings_df.to_dict(orient="records"),
        "bot_winrate_bucket": bot_bucket_df.to_dict(orient="records"),
        "bot_unit": bot_unit_df.to_dict(orient="records"),
        "bot_unit_bucket": bot_bucket_unit_df.to_dict(orient="records"),
        "support_per_bot": support_per_bot_df.to_dict(orient="records"),
        "support_winrate": support_winrate_df.to_dict(orient="records"),
    },
    "games_raw": games_df.to_dict(orient="records"),
    "player_game_units": pg.to_dict(orient="records"),
    "bot_player_game_units": bot_pg.to_dict(orient="records"),
    "support_unit_impact": support_df.to_dict(orient="records"),
    "parsed_replays": parsed_digest,
}
with open(out / "analysis_bundle.json", "w") as f:
    json.dump(bundle, f, indent=2, default=str)

print("Wrote raw data + bundle:")
for p in sorted(out.glob("*.csv")) + sorted(out.glob("*.json")):
    sz = p.stat().st_size
    print(f"  {p.name:40s}  {sz:>10,} bytes")

# Zip the whole run directory and offer Colab download for offline analysis
zip_base = str(out) + "_bundle"
zip_path = shutil.make_archive(zip_base, "zip", root_dir=out)
print(f"\nZipped run -> {zip_path}")

try:
    from google.colab import files as colab_files

    colab_files.download(zip_path)
    print("Colab download triggered.")
except (ImportError, ModuleNotFoundError):
    print("Not in Colab - to download locally, copy:", zip_path)

## Interpreting the results

**First-turn advantage**:
- If `p1_score` overlaps 0.50 in its 95% CI for every bucket, there's no measurable first-turn advantage at this sample size — increase `GAMES_PER_SIDE`.
- If `p1_score > 0.55` only in the `small` bucket, the warrior-rush hypothesis is supported. Either a P2 starting-gold offset or a free P2 starting unit is the natural fix.
- High `draw_rate` (>30%) indicates `MAX_TURNS` is being hit. Either raise it or tune game mechanics so games resolve.

**Unit competitiveness**:
- A unit with high `prod_share` *and* positive `delta` in the win/loss table is probably overtuned — players build it a lot *and* it correlates with winning.
- A unit with `pct_player_games_built` near 0 is unused. Either too expensive, too weak, or strictly dominated by another unit at its price point.
- Compare `win_gold_share` vs `loss_gold_share` per bucket — gaps > 0.05 are interesting; gaps > 0.10 are strong signals.

**Suggested experiments** (set in `ENV_OVERRIDES`, rerun with a fresh `RUN_TAG`, compare bundles across runs):
- `{"unit_cost": {"W": 225}}` — bump warrior cost. Headline: does small-map P1 winrate drop and does Archer/Knight share rise?
- `{"starting_gold": {1: 250, 2: 300}}` — komi-style gold offset. Headline: does P1 winrate move toward 0.50?
- `{"starting_unit": {2: "W"}}` — free P2 warrior on HQ. Stronger than gold (immediate board presence). Try `"A"` or `"K"` for variations.
- `{"starting_unit": {2: "W"}, "starting_gold": {1: 250, 2: 200}}` — combined: free P2 unit but reduced P2 gold, to test whether a unit is worth more than 50 gold of tempo.
- `{"unit_stat": {"A": {"defence": 2}}}` — buff Archer survivability. Headline: does Archer share rise in winning comps?
- `{"income": {"headquarters": 130}}` — slow economy. Headline: does the game shift toward higher-tier units?

**Caveats**:
- Built-in bots (`SimpleBot`, `MediumBot`, `AdvancedBot`) have hand-coded strategies. They reflect both the bots and the game balance. For pure balance signal, also run with LLM/model bots if available.
- 4 games-per-side per map gives ~64 games per matchup at default config — usable for spotting bucket-level effects, weak for fine tuning. Bump to 16+ for narrow CIs.
- Bots may not adapt to `starting_unit` overrides. Hand-coded bots assume fresh-start conditions; the free unit will get used but the bot won't integrate it into a long-term plan. This still measures the raw balance impact, but expect winrate effects to be slightly understated relative to a bot tuned for the new rules.